Import libraries

In [1]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras import layers, models
import numpy as np
from pathlib import Path
import os
import yaml

In [15]:
#### Move up directories until we find the 'data' folder
while not Path("data").exists() and len(Path.cwd().parts) > 1:
    os.chdir("..")
print(f"Working directory set to: {os.getcwd()}")


Working directory set to: E:\CS\SEM 6\DL\Mini Project\Trash-Classification-System


In [3]:
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

SPLITS_CSV = Path(cfg["data"]["splits_csv"]) 
#NUM_CLASSES = cfg["data"]["num_classes"]
#BATCH_SIZE = 32 
#LR = 1e-4

In [4]:
df = pd.read_csv(SPLITS_CSV)

train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
test_df  = df[df["split"] == "test"]

Parameters (MATCH TEAM SETTINGS)

In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

Function to load images

In [6]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0
    return img, label

Create TensorFlow datasets

In [7]:
def create_dataset(df):
    paths = df["filepath"].values
    labels = df["class_idx"].values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda x, y: load_image(x, y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    return ds

train_ds = create_dataset(train_df)
val_ds   = create_dataset(val_df)
test_ds  = create_dataset(test_df)

Data augmentation

In [8]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

Build MobileNetV3-Large model

In [9]:
base_model = MobileNetV3Large(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(6, activation='softmax')
])

Compile

In [10]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

Train

In [11]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=7
)

Epoch 1/7
329/329 ━━━━━━━━━━━━━━━━━━━━ 294s 866ms/step - accuracy: 0.4439 - loss: 1.4596 - val_accuracy: 0.4569 - val_loss: 1.4481
Epoch 2/7
329/329 ━━━━━━━━━━━━━━━━━━━━ 273s 829ms/step - accuracy: 0.4722 - loss: 1.3723 - val_accuracy: 0.5049 - val_loss: 1.3119
Epoch 3/7
329/329 ━━━━━━━━━━━━━━━━━━━━ 275s 837ms/step - accuracy: 0.4842 - loss: 1.3302 - val_accuracy: 0.5027 - val_loss: 1.3254
Epoch 4/7
329/329 ━━━━━━━━━━━━━━━━━━━━ 283s 859ms/step - accuracy: 0.4955 - loss: 1.3045 - val_accuracy: 0.5142 - val_loss: 1.2993
Epoch 5/7
329/329 ━━━━━━━━━━━━━━━━━━━━ 302s 919ms/step - accuracy: 0.5046 - loss: 1.2857 - val_accuracy: 0.5231 - val_loss: 1.2536
Epoch 6/7
329/329 ━━━━━━━━━━━━━━━━━━━━ 303s 920ms/step - accuracy: 0.5088 - loss: 1.2692 - val_accuracy: 0.5218 - val_loss: 1.2617
Epoch 7/7
329/329 ━━━━━━━━━━━━━━━━━━━━ 302s 919ms/step - accuracy: 0.5132 - loss: 1.2567 - val_accuracy: 0.5258 - val_loss: 1.2443


Fine-tuning (IMPORTANT)

In [12]:
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train_ds, validation_data=val_ds, epochs=5)

Epoch 1/5
329/329 ━━━━━━━━━━━━━━━━━━━━ 1287s 4s/step - accuracy: 0.3885 - loss: 2.6864 - val_accuracy: 0.4302 - val_loss: 2.9404
Epoch 2/5
329/329 ━━━━━━━━━━━━━━━━━━━━ 1252s 4s/step - accuracy: 0.5130 - loss: 1.7864 - val_accuracy: 0.2022 - val_loss: 2.8824
Epoch 3/5
329/329 ━━━━━━━━━━━━━━━━━━━━ 1227s 4s/step - accuracy: 0.5710 - loss: 1.4407 - val_accuracy: 0.2236 - val_loss: 3.1508
Epoch 4/5
329/329 ━━━━━━━━━━━━━━━━━━━━ 1484s 5s/step - accuracy: 0.6177 - loss: 1.2260 - val_accuracy: 0.2480 - val_loss: 3.5204
Epoch 5/5
329/329 ━━━━━━━━━━━━━━━━━━━━ 1300s 4s/step - accuracy: 0.6563 - loss: 1.0798 - val_accuracy: 0.3578 - val_loss: 3.3411


Test accuracy

In [13]:
loss, acc = model.evaluate(test_ds)
print("Test Accuracy:", acc)
print("Test Loss:", loss)

71/71 ━━━━━━━━━━━━━━━━━━━━ 47s 647ms/step - accuracy: 0.3431 - loss: 3.3359
Test Accuracy: 0.3431110978126526
Test Loss: 3.3359439373016357


Save model

In [14]:
model.save("models/mobilenetv3_test1.h5")